## Libraries

In [1]:
from config.settings import DATABASE
from pyspark.sql import SparkSession
from mysql import connector

In [2]:
# Instance SparkSesssion
spark = SparkSession.builder\
    .appName('Processing crypto_hub_db') \
    .master('local[*]') \
    .getOrCreate()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 53978)


## Extract data from database

In [3]:
# Create connection
connection = connector.connect(**DATABASE)
cursor = connection.cursor()

# List tables
tables = ['cryptos', 'market_bitcoin', 'historical_bitcoin_prices']

# Dictionary for storage DataFrames
dataframes = {}

# Iterar sobre cada tabla y cargarla en PySpark
for table in tables:
    query = f"SELECT * FROM {table}"
    cursor.execute(query)

    # Obtener los resultados y nombres de columnas
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]

    # Convertir a DataFrame de PySpark
    df = spark.createDataFrame(rows, schema=columns)
    dataframes[table] = df

    print(f"Tabla {table} cargada correctamente.")

# Cerrar conexión
cursor.close()
connection.close()

Tabla cryptos cargada correctamente.
Tabla market_bitcoin cargada correctamente.
Tabla historical_bitcoin_prices cargada correctamente.


## Save origin data

In [15]:
# Write tables 
dataframes['cryptos'].write.csv('data/raw/cryptos', header=True, mode='overwrite', sep=',')
dataframes['market_bitcoin'].write.csv('data/raw/market_bitcoin', header=True, mode='overwrite', sep=',')
dataframes['historical_bitcoin_prices'].write.csv('data/raw/historical_bitcoin_prices', header=True, mode='overwrite', sep=',')
print(f'The tables {", ".join(dataframes.keys())} has beer written...')

The tables: cryptos, market_bitcoin, historical_bitcoin_prices has beer written...
